# Spark Dataframe

## 1. Connection

In [1]:
from faker import Faker
from pyspark.sql import SparkSession

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/tushar/pyspark-labs/.venv/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/tushar/pyspark-labs/.venv/lib/python3.10/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/home/tushar/pyspark-labs/.venv/lib/python3.10/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
  File "/home/tushar/pyspark-labs/.venv/lib/python3.10/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/tushar/pyspark-labs/.venv/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.resu

Spark Connect is a client-server architecture within Apache Spark that enables remote connectivity to Spark clusters from any application. PySpark provides the client for the Spark Connect server, allowing Spark to be used as a service.

Pyspark also needs jvm, spark master and workers run their own jvm

`SparkSession.builder.master()`
This is the classic Spark API. It tells the Spark driver where to submit jobs.
The driver (your Python process) runs on your machine, connects to the Spark Master, and schedules tasks on the workers.


`SparkSession.builder.remote()` is newer and is used with Spark Connect.
Instead of embedding the Spark driver inside your Python process, your Python code becomes a thin client. The actual Spark driver runs remotely in a Spark Connect server.


Python in worker has different version: 3.10 than that in driver: 3.12, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set
Pyspark driver python version and spark worker python version should be same

In [2]:
# test that connection works

spark = (
    SparkSession.builder
    .master("spark://localhost:7077")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9100")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .appName("pyspark-labs")
    .getOrCreate()
)

print(spark.version)
spark.range(5).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/26 13:12:29 WARN Utils: Your hostname, oscar, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/26 13:12:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/tushar/pyspark-labs/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/tushar/.ivy2.5.2/cache
The jars for the packages stored in: /home/tushar/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6baf8c06-7372-4c59-987c-c98ef5b5b7d7;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found software.amazon.awssdk#bundle;2.24.6 in central
	found org.wildfly.openssl#wildfly-openssl;1.1.3.Final in central
:: resolution report :: resolve 298ms :: artifa

4.0.1


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



## 2. Creating dataframe

In [3]:
# Dataframe
from pyspark.sql import Row
from random import randint

fake = Faker()

In [4]:
def generate_user_data(n):
    """Generate and return user data one at a time."""
    for _ in range(n):
        yield {
            "name": fake.name(),
            "email": fake.email(),
            "phone_number": fake.phone_number(),
            "job": fake.job(),
            "date_of_birth": fake.date_of_birth(),
            "company": fake.company(),
            "address": fake.address(),
            "ssn": fake.ssn(),
            "state": fake.state(),
            "country": fake.country(),
            "salary": randint(50_000, 200_000)
        }

In [ ]:

n = 100

df = spark.createDataFrame([
    Row(name=user["name"],
        email=user["email"],
        phone_number=user["phone_number"],
        job=user["job"],
        date_of_birth=user["date_of_birth"],
        company=user["company"],
        address=user["address"],
        ssn=user["ssn"],
        state=user["state"],
        country=user["country"],
        salary=user["salary"]
    )
    for user in generate_user_data(n)
])
df.show()

+-----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+------------+--------------------+------+
|             name|               email|        phone_number|                 job|date_of_birth|             company|             address|        ssn|       state|             country|salary|
+-----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+------------+--------------------+------+
|Kimberly Mccarthy|harrellfred@examp...|001-306-453-6130x901|Development worke...|   1950-06-26|          Burton Inc|6586 Ian Prairie ...|600-36-2688|    Michigan|               Nepal|100930|
|      Jacob Gates|andrewpowers@exam...|  (914)359-6287x7808|Environmental edu...|   1949-06-28|           Burke LLC|403 Rachael Knoll...|286-50-0882|      Kansas|United States Vir...| 99892|
|   Michael Kelley|mariamartinez@exa...|

In [6]:
# Create a PySpark DataFrame with an explicit schema.
n = 100

df = spark.createDataFrame(
    [
        Row(
            name=user["name"],
            email=user["email"],
            phone_number=user["phone_number"],
            job=user["job"],
            date_of_birth=user["date_of_birth"],
            company=user["company"],
            address=user["address"],
            ssn=user["ssn"],
            state=user["state"],
            country=user["country"],
            salary=user["salary"]
        ) for user in generate_user_data(n)
    ],
    schema="name string, email string, phone_number string, job string, date_of_birth date, company string, address string, ssn string, state string, country string, salary int"
)
# string, int, date, timestamp, boolean
df.show()


+-----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+--------------+--------------------+------+
|             name|               email|        phone_number|                 job|date_of_birth|             company|             address|        ssn|         state|             country|salary|
+-----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+--------------+--------------------+------+
|     Lindsay Rich| jcooper@example.net|        222-666-8179|Public relations ...|   1993-05-15|       Schwartz-Khan|5436 Corey Union ...|571-16-8333|        Alaska|             Bolivia| 98455|
|   Stephen Murphy|sierraowens@examp...|   471.454.4552x9042|           Paramedic|   2001-02-06|     Larson and Sons|7749 Contreras Es...|119-28-3013|      Virginia|                Chad|182048|
|     Joseph Arias| heidi36@ex

In [7]:
# Create a PySpark DataFrame from a pandas DataFrame
import pandas as pd

In [8]:
pdf = pd.DataFrame([user for user in generate_user_data(n)])
print(pdf.shape)
df = spark.createDataFrame(pdf)
df.show()

(100, 11)
+----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+--------------+--------------------+------+
|            name|               email|        phone_number|                 job|date_of_birth|             company|             address|        ssn|         state|             country|salary|
+----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+--------------+--------------------+------+
|    Edward Duran|  smoore@example.com|  (940)368-9596x8966|            Musician|   1932-06-13|         Garrett Inc|783 Michael Way S...|023-39-2315|      Virginia|          Martinique| 88791|
|        Tara Cox|  paul70@example.net|        798.216.6285|      Retail manager|   1923-07-17|        Miller-Perez|57349 Robin Views...|775-79-1134|      Kentucky|            Anguilla| 65638|
|   Crystal Clark|yarmstr

## 3. Viewing data

In [9]:
# All DataFrames above result same.
df.show()
df.printSchema()

+----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+--------------+--------------------+------+
|            name|               email|        phone_number|                 job|date_of_birth|             company|             address|        ssn|         state|             country|salary|
+----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+--------------+--------------------+------+
|    Edward Duran|  smoore@example.com|  (940)368-9596x8966|            Musician|   1932-06-13|         Garrett Inc|783 Michael Way S...|023-39-2315|      Virginia|          Martinique| 88791|
|        Tara Cox|  paul70@example.net|        798.216.6285|      Retail manager|   1923-07-17|        Miller-Perez|57349 Robin Views...|775-79-1134|      Kentucky|            Anguilla| 65638|
|   Crystal Clark|yarmstrong@exampl

In [10]:
# The top rows of a DataFrame can be displayed using DataFrame.show().

df.show(1, truncate=False)  # Show the first row without truncating the columns.
df.show(1, truncate=False, vertical=True)  # Show the first row vertically.

+------------+------------------+------------------+--------+-------------+-----------+--------------------------------------------------+-----------+--------+----------+------+
|name        |email             |phone_number      |job     |date_of_birth|company    |address                                           |ssn        |state   |country   |salary|
+------------+------------------+------------------+--------+-------------+-----------+--------------------------------------------------+-----------+--------+----------+------+
|Edward Duran|smoore@example.com|(940)368-9596x8966|Musician|1932-06-13   |Garrett Inc|783 Michael Way Suite 881\nKatherinebury, MS 74070|023-39-2315|Virginia|Martinique|88791 |
+------------+------------------+------------------+--------+-------------+-----------+--------------------------------------------------+-----------+--------+----------+------+
only showing top 1 row
-RECORD 0-----------------------------------------------------------
 name          | E

In [11]:
# schema
print(df.columns)
df.printSchema()

['name', 'email', 'phone_number', 'job', 'date_of_birth', 'company', 'address', 'ssn', 'state', 'country', 'salary']
root
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- job: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- company: string (nullable = true)
 |-- address: string (nullable = true)
 |-- ssn: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- salary: long (nullable = true)



In [12]:
# summary of dataframe
df.select("name", "email", "date_of_birth").describe().show(truncate=False)

+-------+-------------+------------------------+
|summary|name         |email                   |
+-------+-------------+------------------------+
|count  |100          |100                     |
|mean   |NULL         |NULL                    |
|stddev |NULL         |NULL                    |
|min    |Abigail Brown|abbottjustin@example.net|
|max    |William Mccoy|ysullivan@example.net   |
+-------+-------------+------------------------+



In [13]:
df.select("name", "email", "date_of_birth").summary().show(truncate=False) # can also use summary instead of describe

+-------+-------------+------------------------+
|summary|name         |email                   |
+-------+-------------+------------------------+
|count  |100          |100                     |
|mean   |NULL         |NULL                    |
|stddev |NULL         |NULL                    |
|min    |Abigail Brown|abbottjustin@example.net|
|25%    |NULL         |NULL                    |
|50%    |NULL         |NULL                    |
|75%    |NULL         |NULL                    |
|max    |William Mccoy|ysullivan@example.net   |
+-------+-------------+------------------------+



In [14]:
# collect, take and tail
# DataFrame.collect() collects the distributed data to the driver side as the local data in Python. Note that this can throw an out-of-memory error when the dataset is too large to fit in the driver side because it collects all the data from executors to the driver side.
df.collect()


[Row(name='Edward Duran', email='smoore@example.com', phone_number='(940)368-9596x8966', job='Musician', date_of_birth=datetime.date(1932, 6, 13), company='Garrett Inc', address='783 Michael Way Suite 881\nKatherinebury, MS 74070', ssn='023-39-2315', state='Virginia', country='Martinique', salary=88791),
 Row(name='Tara Cox', email='paul70@example.net', phone_number='798.216.6285', job='Retail manager', date_of_birth=datetime.date(1923, 7, 17), company='Miller-Perez', address='57349 Robin Views\nJonathanburgh, UT 58332', ssn='775-79-1134', state='Kentucky', country='Anguilla', salary=65638),
 Row(name='Crystal Clark', email='yarmstrong@example.com', phone_number='+1-770-625-3792', job='Associate Professor', date_of_birth=datetime.date(1920, 5, 6), company='Carpenter-Ruiz', address='16336 Amanda Plaza Suite 534\nKellyview, CA 69342', ssn='894-21-7731', state='New Jersey', country='South Georgia and the South Sandwich Islands', salary=64523),
 Row(name='Dorothy Shannon', email='orodrigue

In [15]:
df.take(5)  # Returns the first n rows as a list of Row.

[Row(name='Edward Duran', email='smoore@example.com', phone_number='(940)368-9596x8966', job='Musician', date_of_birth=datetime.date(1932, 6, 13), company='Garrett Inc', address='783 Michael Way Suite 881\nKatherinebury, MS 74070', ssn='023-39-2315', state='Virginia', country='Martinique', salary=88791),
 Row(name='Tara Cox', email='paul70@example.net', phone_number='798.216.6285', job='Retail manager', date_of_birth=datetime.date(1923, 7, 17), company='Miller-Perez', address='57349 Robin Views\nJonathanburgh, UT 58332', ssn='775-79-1134', state='Kentucky', country='Anguilla', salary=65638),
 Row(name='Crystal Clark', email='yarmstrong@example.com', phone_number='+1-770-625-3792', job='Associate Professor', date_of_birth=datetime.date(1920, 5, 6), company='Carpenter-Ruiz', address='16336 Amanda Plaza Suite 534\nKellyview, CA 69342', ssn='894-21-7731', state='New Jersey', country='South Georgia and the South Sandwich Islands', salary=64523),
 Row(name='Dorothy Shannon', email='orodrigue

In [16]:
df.tail(5)  # Returns the last n rows as a list of Row objects.

[Row(name='Denise Davis', email='simmonsmegan@example.org', phone_number='+1-618-550-2080', job='Advertising copywriter', date_of_birth=datetime.date(1942, 1, 29), company='Horton, Mcmillan and Mitchell', address='9806 Gordon Drives Apt. 987\nJamesstad, IN 89966', ssn='878-23-9110', state='Arizona', country='Myanmar', salary=87205),
 Row(name='Dawn Lewis', email='cbrown@example.org', phone_number='2188494645', job='Horticulturist, commercial', date_of_birth=datetime.date(1973, 8, 4), company='Hill, Blair and Thompson', address='628 Brittany Pike\nWest Christian, MA 74212', ssn='885-04-2032', state='Vermont', country='Libyan Arab Jamahiriya', salary=81693),
 Row(name='Andrea Craig', email='ayalawilliam@example.com', phone_number='+1-788-917-1292x851', job='Media buyer', date_of_birth=datetime.date(1954, 10, 5), company='Reed, Garza and Taylor', address='286 Dunlap Meadows Suite 540\nPort Sandraview, CO 72408', ssn='425-25-7817', state='Ohio', country='Philippines', salary=97345),
 Row(n

In [17]:
# spark dataframe to pandas
# PySpark DataFrame also provides the conversion back to a pandas DataFrame to leverage pandas API. Note that toPandas also collects all data into the driver side that can easily cause an out-of-memory-error when the data is too large to fit into the driver side.
pdf = df.toPandas()
print(type(pdf))
pdf

<class 'pandas.core.frame.DataFrame'>


,name,email,phone_number,job,date_of_birth,company,address,ssn,state,country,salary
0,Edward Duran,smoore@example.com,(940)368-9596x8966,Musician,1932-06-13,Garrett Inc,"783 Michael Way Suite 881\nKatherinebury, MS 7...",023-39-2315,Virginia,Martinique,88791
1,Tara Cox,paul70@example.net,798.216.6285,Retail manager,1923-07-17,Miller-Perez,"57349 Robin Views\nJonathanburgh, UT 58332",775-79-1134,Kentucky,Anguilla,65638
2,Crystal Clark,yarmstrong@example.com,+1-770-625-3792,Associate Professor,1920-05-06,Carpenter-Ruiz,"16336 Amanda Plaza Suite 534\nKellyview, CA 69342",894-21-7731,New Jersey,South Georgia and the South Sandwich Islands,64523
3,Dorothy Shannon,orodriguez@example.com,+1-901-473-6217x8056,Pensions consultant,2015-05-17,Jackson-Flowers,"8399 Melvin Locks Apt. 116\nPort Candace, DE 9...",070-67-0651,Missouri,Northern Mariana Islands,56390
4,Dr. Meagan Baker,michele07@example.org,9456766263,Diagnostic radiographer,1986-06-16,Holmes-Moreno,"9217 Joshua Club\nMatthewside, DE 08831",108-05-1333,Kansas,Djibouti,101589
...,...,...,...,...,...,...,...,...,...,...,...
95,Denise Davis,simmonsmegan@example.org,+1-618-550-2080,Advertising copywriter,1942-01-29,"Horton, Mcmillan and Mitchell","9806 Gordon Drives Apt. 987\nJamesstad, IN 89966",878-23-9110,Arizona,Myanmar,87205
96,Dawn Lewis,cbrown@example.org,2188494645,"Horticulturist, commercial",1973-08-04,"Hill, Blair and Thompson","628 Brittany Pike\nWest Christian, MA 74212",885-04-2032,Vermont,Libyan Arab Jamahiriya,81693
97,Andrea Craig,ayalawilliam@example.com,+1-788-917-1292x851,Media buyer,1954-10-05,"Reed, Garza and Taylor","286 Dunlap Meadows Suite 540\nPort Sandraview,...",425-25-7817,Ohio,Philippines,97345
98,Victoria Bender,toddmiller@example.net,(961)300-7411,"Doctor, general practice",1948-10-24,Mendez-Willis,"77499 Peter Forks Apt. 293\nEast Davidville, W...",686-95-2232,Maine,Colombia,182525


## 4. Selecting and Accessing Data


In [18]:
# PySpark DataFrame is lazily evaluated and simply selecting a column does not trigger the computation but it returns a Column instance.
df.name

Column<'name'>

In [19]:
# actually, most of operations on columns return the column instance and not the actual data.
from pyspark.sql import Column
from pyspark.sql.functions import upper

type(df.name), type(df.address), type(upper(df.address))

(pyspark.sql.classic.column.Column,
 pyspark.sql.classic.column.Column,
 pyspark.sql.classic.column.Column)

In [20]:
# These Columns can be used to select the columns from a DataFrame. For example, DataFrame.select() takes the Column instances that returns another DataFrame.
df.select(df.name, df.address).show()

+----------------+--------------------+
|            name|             address|
+----------------+--------------------+
|    Edward Duran|783 Michael Way S...|
|        Tara Cox|57349 Robin Views...|
|   Crystal Clark|16336 Amanda Plaz...|
| Dorothy Shannon|8399 Melvin Locks...|
|Dr. Meagan Baker|9217 Joshua Club\...|
|     James Gomez|146 Amy Circles\n...|
|Christopher Wong|84586 Walker Ranc...|
|   Chelsea Jones|91644 Mathis Cent...|
|   Eric Mcmillan|853 Lawrence Squa...|
|   Meghan Garner|408 Davis Manor\n...|
|  Roberto Herman|522 Zimmerman Wal...|
|    Tim Thompson|754 Rosario Hill\...|
|  Tracy Martinez|804 Daisy Express...|
|  Matthew Mccall|096 Hernandez Cro...|
|   Walter Miller|7978 David Spur S...|
|   Marissa Green|Unit 3595 Box 316...|
|Nicole Maldonado|PSC 8692, Box 318...|
|      Mark Meyer|530 Shepherd Fiel...|
|Alyssa Rodriguez|72120 Wall Mount\...|
| Mitchell Carter|12197 Christian G...|
+----------------+--------------------+
only showing top 20 rows


In [21]:
# Assign new Column instance.
df.withColumn("upper_address", upper(df.address)).show()  # Add a new column with the upper case of address.

+----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+--------------+--------------------+------+--------------------+
|            name|               email|        phone_number|                 job|date_of_birth|             company|             address|        ssn|         state|             country|salary|       upper_address|
+----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+--------------+--------------------+------+--------------------+
|    Edward Duran|  smoore@example.com|  (940)368-9596x8966|            Musician|   1932-06-13|         Garrett Inc|783 Michael Way S...|023-39-2315|      Virginia|          Martinique| 88791|783 MICHAEL WAY S...|
|        Tara Cox|  paul70@example.net|        798.216.6285|      Retail manager|   1923-07-17|        Miller-Perez|57349 Robin Views...|775-79-

In [22]:
# Filter data in dataframe
# To select a subset of rows, use DataFrame.filter().
# use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.
df.filter(df.name == 'Lisa Malone').show()  
df.filter(df.name.contains('John')).show()  
df.filter(df.name.startswith("A") & df.name.endswith("son")).show()  # Filter rows where name starts with "A" and ends with "son".

+----+-----+------------+---+-------------+-------+-------+---+-----+-------+------+
|name|email|phone_number|job|date_of_birth|company|address|ssn|state|country|salary|
+----+-----+------------+---+-------------+-------+-------+---+-----+-------+------+
+----+-----+------------+---+-------------+-------+-------+---+-----+-------+------+



+------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+------------+----------------+------+
|        name|               email|        phone_number|                 job|date_of_birth|             company|             address|        ssn|       state|         country|salary|
+------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+------------+----------------+------+
|Juan Johnson|lauren26@example.org|          3277848730|      Health visitor|   1949-02-19|Lewis, Conley and...|736 Luke Knoll\nT...|181-40-0680|    Kentucky|           China| 92663|
| John Willis| jason62@example.com|          9029879223|Medical technical...|   1957-04-14|    Johnson and Sons|276 Billy Neck\nK...|511-66-1734|Rhode Island|     Saint Lucia| 51198|
| John Flores|erodriguez@exampl...|+1-470-634-9307x8...|Rural practice su...|   1986-

## 5. Applying a function

In [23]:
# PySpark supports various UDFs and APIs to allow users to execute Python native functions. See also the latest Pandas UDFs and Pandas Function APIs. For instance, the example below allows users to directly use the APIs in a pandas Series within Python native function.
from pyspark.sql.functions import pandas_udf


@pandas_udf("string")
def pandas_append_a(series: pd.Series) -> pd.Series:
    """Append 'a' to each element in the pandas Series."""
    return series + ' a'

df.select(pandas_append_a(df.name)).show()  # Apply the pandas UDF to the 'name' column and show the result.
# this required to install pyarrow in the spark worker docker container (modified the base image via spark.Dockerfile)

+---------------------+
|pandas_append_a(name)|
+---------------------+
|       Edward Duran a|
|           Tara Cox a|
|      Crystal Clark a|
|    Dorothy Shannon a|
|   Dr. Meagan Baker a|
|        James Gomez a|
|   Christopher Wong a|
|      Chelsea Jones a|
|      Eric Mcmillan a|
|      Meghan Garner a|
|     Roberto Herman a|
|       Tim Thompson a|
|     Tracy Martinez a|
|     Matthew Mccall a|
|      Walter Miller a|
|      Marissa Green a|
|   Nicole Maldonado a|
|         Mark Meyer a|
|   Alyssa Rodriguez a|
|    Mitchell Carter a|
+---------------------+
only showing top 20 rows


In [24]:
# Another example is DataFrame.mapInPandas which allows users directly use the APIs in a pandas DataFrame without any restrictions such as the result length.

# def pandas_filter_function(pdf: pd.DataFrame) -> pd.DataFrame:
#     """Filter rows in the pandas DataFrame where the 'name' column contains 'John'."""
#     return pdf[pdf['name'].str.contains('John')]

def pandas_filter_function(iterator):
    for pandas_df in iterator:
        # yield pandas_df[pandas_df['name'].str.contains('John')]
        yield pandas_df[pandas_df.name == "Kevin Avila"]  # Filter rows where name is "Kevin Avila".

df.mapInPandas(pandas_filter_function, schema=df.schema).show()  # Apply the pandas function to the DataFrame and show the result.
# mapInPandas lets you apply a function that transforms one Pandas DataFrame into another, operating per-partition, with full control over row filtering/transformation using pandas/numpy operations instead of Spark's built-in functions.

# What actually happens under the hood:

# Partitioning: spark_df is split across partitions (same as any Spark operation). Each partition lives on some executor.
# Per-partition conversion to Pandas: For each partition, Spark converts the partition's rows into one or more Pandas DataFrames — via Arrow, not row-by-row Python serialization, which is why it's fast. It hands your function an iterator of Pandas DataFrames, not a single DataFrame for the whole dataset.
# Your function runs once per batch, per partition.

+----+-----+------------+---+-------------+-------+-------+---+-----+-------+------+
|name|email|phone_number|job|date_of_birth|company|address|ssn|state|country|salary|
+----+-----+------------+---+-------------+-------+-------+---+-----+-------+------+
+----+-----+------------+---+-------------+-------+-------+---+-----+-------+------+



## 6. Grouping data

In [25]:
# PySpark DataFrame also provides a way of handling grouped data by using the common approach, split-apply-combine strategy. It groups the data by a certain condition applies a function to each group and then combines them back to the DataFrame.

# Grouping and then applying the avg() function to the resulting groups.
df.groupBy("country").avg("salary").show()

+--------------------+------------------+
|             country|       avg(salary)|
+--------------------+------------------+
|            Anguilla|           65638.0|
|            Djibouti|          101589.0|
|Northern Mariana ...|          122313.5|
|          Martinique|           88791.0|
|South Georgia and...|           89961.5|
|             Estonia|          195275.0|
|               Yemen|          181019.0|
|      Norfolk Island|          179989.0|
|               China|103396.66666666667|
|          Cape Verde|           51697.0|
|                Oman|          117411.0|
|            Botswana|          106778.0|
|               Congo|           83748.0|
|             Georgia|          160391.0|
|      Czech Republic|          185490.0|
|            Tanzania|          127563.5|
|               Nepal|           82118.0|
|                Niue|          126470.0|
|Turks and Caicos ...|          192438.0|
|          Guadeloupe|           53801.0|
+--------------------+------------

In [26]:
# You can also apply a Python native function against each group by using pandas API.
def plus_mean(pandas_df: pd.DataFrame) -> pd.DataFrame:
    """Update salary in pandas DataFrame with the sum of 'salary' and the mean of 'salary'."""
    return pandas_df.assign(salary=pandas_df["salary"]+pandas_df["salary"].mean())

df.groupBy("country").applyInPandas(plus_mean, schema=df.schema).show()  # Apply the pandas function to each group and show the result.

+-----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+------------+--------------------+------+
|             name|               email|        phone_number|                 job|date_of_birth|             company|             address|        ssn|       state|             country|salary|
+-----------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+-----------+------------+--------------------+------+
|    Rebecca Smith| jesse92@example.com|001-566-826-2118x...|      Hydrogeologist|   2019-07-31|        Smith-Gordon|546 Brown Lodge S...|462-67-0032|    New York|             Albania|197312|
|    Cody Sullivan|joshua01@example.net| +1-887-284-3990x538|Accommodation man...|   2015-12-06|        Hill-Escobar|1025 Heidi Rapids...|635-78-5882|   Tennessee|             Andorra|123590|
|         Tara Cox|  paul70@example.net|

In [27]:
# cogrouping
df1 = spark.createDataFrame(
    [(20000101, 1, 1.0), (20000101, 2, 2.0), (20000102, 1, 3.0), (20000102, 2, 4.0)],
    ('time', 'id', 'v1'))

df2 = spark.createDataFrame(
    [(20000101, 1, 'x'), (20000101, 2, 'y')],
    ('time', 'id', 'v2'))

def merge_ordered(l, r):
    return pd.merge_ordered(l, r)

df1.groupby('id').cogroup(df2.groupby('id')).applyInPandas(
    merge_ordered, schema='time int, id int, v1 double, v2 string').show()

# For id=1, it pairs df1's id=1 rows with df2's id=1 rows. Same for id=2. If a key exists in one side but not the other, you still get a pair — just with an empty pandas DataFrame on the missing side.
# .applyInPandas(merge_ordered, schema=...) runs your function once per key, receiving both sides as separate pandas DataFrames

+--------+---+---+----+
|    time| id| v1|  v2|
+--------+---+---+----+
|20000101|  1|1.0|   x|
|20000102|  1|3.0|NULL|
|20000101|  2|2.0|   y|
|20000102|  2|4.0|NULL|
+--------+---+---+----+



## 7. Getting data in/out

In [28]:
# CSV is straightforward and easy to use.
# Parquet and ORC are efficient and compact file formats to read and write faster.
# other data sources available in PySpark such as JDBC, text, binaryFile, Avro, etc

In [29]:
# csv
# df.write.csv('/data/foo.csv', header=True)
# spark.read.csv('foo.csv', header=True).show()


df.write.csv('s3a://pyspark-labs/foo.csv', header=True, mode='overwrite')

# df2 = spark.read.csv('s3a://pyspark-labs/foo.csv', header=True)
# df2.show()

26/07/26 13:13:58 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/07/26 13:13:59 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/07/26 13:28:51 WARN TaskSetManager: Lost task 11.0 in stage 50.0 (TID 255) (172.19.0.5 executor 1): java.net.ConnectException: getFileStatus on s3a://pyspark-labs/foo.csv/_temporary/0/_temporary/attempt_202607261313599207888619395794931_0050_m_000011_255/part-00011-dd997db9-32d3-409c-b691-073abbd4f97e-c000.csv: software.amazon.awssdk.core.exception.SdkClientException: Unable to execute HTTP request: Connect to localhost:9100 [localhost/127.0.0.1, localhost/0:0:0:0:0:0:0:1] failed: Connection refused

Py4JJavaError: An error occurred while calling o337.csv.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 4 in stage 50.0 failed 4 times, most recent failure: Lost task 4.3 in stage 50.0 (TID 292) (172.19.0.6 executor 0): java.net.ConnectException: getFileStatus on s3a://pyspark-labs/foo.csv/_temporary/0/_temporary/attempt_202607261313599207888619395794931_0050_m_000004_292/part-00004-dd997db9-32d3-409c-b691-073abbd4f97e-c000.csv: software.amazon.awssdk.core.exception.SdkClientException: Unable to execute HTTP request: Connect to localhost:9100 [localhost/127.0.0.1, localhost/0:0:0:0:0:0:0:1] failed: Connection refused:    software.amazon.awssdk.core.exception.SdkClientException: Unable to execute HTTP request: Connect to localhost:9100 [localhost/127.0.0.1, localhost/0:0:0:0:0:0:0:1] failed: Connection refused: Connection refused
	at java.base/jdk.internal.reflect.GeneratedConstructorAccessor46.newInstance(Unknown Source)
	at java.base/jdk.internal.reflect.Del